In [ ]:
import os
import random
import shutil
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

def split_datasets_resumable(source_dir_jpg, source_dir_tif, dest_dir_jpg, dest_dir_tif,
                             ext_jpg='.jpg', ext_tif='.tif',
                             train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42):

    # Ensure ratios make sense
    if round(train_ratio + val_ratio + test_ratio, 5) != 1.0:
        raise ValueError("Train, val, and test ratios must sum to exactly 1.0")

    # Set the seed for reproducibility across restarts
    random.seed(random_seed)

    src_jpg = Path(source_dir_jpg)
    src_tif = Path(source_dir_tif)
    dst_jpg = Path(dest_dir_jpg)
    dst_tif = Path(dest_dir_tif)

    classes = [d.name for d in src_jpg.iterdir() if d.is_dir()]

    if not classes:
        print(f"Error: No folders found in {src_jpg}. Please check your paths.")
        return

    for cls in classes:
        print(f"\nProcessing paired class: {cls}...")

        # Create destination directories
        for split in ['train', 'val', 'test']:
            (dst_jpg / split / cls).mkdir(parents=True, exist_ok=True)
            (dst_tif / split / cls).mkdir(parents=True, exist_ok=True)

        # Sort before shuffling to guarantee deterministic order if restarted
        base_names = sorted([f.stem for f in (src_jpg / cls).iterdir() if f.is_file()])

        # Shuffle
        random.shuffle(base_names)

        # Calculate split indices
        total_files = len(base_names)
        train_idx = int(total_files * train_ratio)
        val_idx = train_idx + int(total_files * val_ratio)

        # Slice the base name list
        train_files = base_names[:train_idx]
        val_files = base_names[train_idx:val_idx]
        test_files = base_names[val_idx:]

        # Helper function with skip-if-exists logic
        def copy_paired_files(file_list, split_name):
            skipped = 0
            copied = 0

            for base_name in file_list:
                # 1. Handle JPG (.jpg)
                file_jpg = src_jpg / cls / f"{base_name}{ext_jpg}"
                dest_jpg_path = dst_jpg / split_name / cls / f"{base_name}{ext_jpg}"

                if file_jpg.exists():
                    if not dest_jpg_path.exists():
                        shutil.copy2(file_jpg, dest_jpg_path)
                        copied += 1
                    else:
                        skipped += 1

                # 2. Handle Multispectral TIF (.tif)
                file_tif = src_tif / cls / f"{base_name}{ext_tif}"
                dest_tif_path = dst_tif / split_name / cls / f"{base_name}{ext_tif}"

                if file_tif.exists():
                    if not dest_tif_path.exists():
                        shutil.copy2(file_tif, dest_tif_path)

            print(f"  -> {split_name.capitalize()}: Copied {copied} new images. Skipped {skipped} already existing images.")

        # Execute copying
        copy_paired_files(train_files, 'train')
        copy_paired_files(val_files, 'val')
        copy_paired_files(test_files, 'test')

    print(f"\nSuccess! Both datasets have been identically split into: {dst_jpg.parent}")

# ==========================================
# --- PATH CONFIGURATION ---
# ==========================================

# Base directory inside your satellite_image folder
base_path = '/content/drive/MyDrive/satellite_image'

# Source Folders
source_eurosat_jpg = f'{base_path}/dataset/EuroSAT'
source_eurosat_tif = f'{base_path}/dataset/EuroSATallBands'

# Destination Folders (creates 'model_dataset' inside satellite_image)
dest_eurosat_jpg = f'{base_path}/model_dataset/EuroSat'
dest_eurosat_tif = f'{base_path}/model_dataset/EuroSatallBands'

# Run Splitter
split_datasets_resumable(
    source_dir_jpg=source_eurosat_jpg,
    source_dir_tif=source_eurosat_tif,
    dest_dir_jpg=dest_eurosat_jpg,
    dest_dir_tif=dest_eurosat_tif,
    ext_jpg='.jpg',
    ext_tif='.tif',
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing paired class: Forest...
  -> Train: Copied 1722 new images. Skipped 374 already existing images.
  -> Val: Copied 449 new images. Skipped 0 already existing images.
  -> Test: Copied 450 new images. Skipped 0 already existing images.

Processing paired class: Highway...
  -> Train: Copied 1746 new images. Skipped 0 already existing images.
  -> Val: Copied 374 new images. Skipped 0 already existing images.
  -> Test: Copied 375 new images. Skipped 0 already existing images.

Processing paired class: Industrial...
  -> Train: Copied 1746 new images. Skipped 0 already existing images.
  -> Val: Copied 374 new images. Skipped 0 already existing images.
  -> Test: Copied 375 new images. Skipped 0 already existing images.

Processing paired class: HerbaceousVegetation...
  -> Train: Copied 2096 new images. Skipped 0 already existing images.
  -> Val: C